# Rock Paper Scissors

Solution for the freeCodeCamp *Machine Learning with Python* project.

The player predicts the opponent's next move with an ensemble of models (Markov chains of order 1-5 on the opponent's history, plus models of the three adaptive bots) and plays the move that beats the prediction.

In [1]:
import random

# DO NOT MODIFY THIS FILE

import random


def play(player1, player2, num_games, verbose=False):
    p1_prev_play = ""
    p2_prev_play = ""
    results = {"p1": 0, "p2": 0, "tie": 0}

    for _ in range(num_games):
        p1_play = player1(p2_prev_play)
        p2_play = player2(p1_prev_play)

        if p1_play == p2_play:
            results["tie"] += 1
            winner = "Tie."
        elif (p1_play == "P" and p2_play == "R") or (
                p1_play == "R" and p2_play == "S") or (p1_play == "S"
                                                       and p2_play == "P"):
            results["p1"] += 1
            winner = "Player 1 wins."
        elif p2_play == "P" and p1_play == "R" or p2_play == "R" and p1_play == "S" or p2_play == "S" and p1_play == "P":
            results["p2"] += 1
            winner = "Player 2 wins."

        if verbose:
            print("Player 1:", p1_play, "| Player 2:", p2_play)
            print(winner)
            print()

        p1_prev_play = p1_play
        p2_prev_play = p2_play

    games_won = results['p2'] + results['p1']

    if games_won == 0:
        win_rate = 0
    else:
        win_rate = results['p1'] / games_won * 100

    print("Final results:", results)
    print(f"Player 1 win rate: {win_rate}%")

    return (win_rate)


def quincy(prev_play, counter=[0]):

    counter[0] += 1
    choices = ["R", "R", "P", "P", "S"]
    return choices[counter[0] % len(choices)]


def mrugesh(prev_opponent_play, opponent_history=[]):
    opponent_history.append(prev_opponent_play)
    last_ten = opponent_history[-10:]
    most_frequent = max(set(last_ten), key=last_ten.count)

    if most_frequent == '':
        most_frequent = "S"

    ideal_response = {'P': 'S', 'R': 'P', 'S': 'R'}
    return ideal_response[most_frequent]


def kris(prev_opponent_play):
    if prev_opponent_play == '':
        prev_opponent_play = "R"
    ideal_response = {'P': 'S', 'R': 'P', 'S': 'R'}
    return ideal_response[prev_opponent_play]


def abbey(prev_opponent_play,
          opponent_history=[],
          play_order=[{
              "RR": 0,
              "RP": 0,
              "RS": 0,
              "PR": 0,
              "PP": 0,
              "PS": 0,
              "SR": 0,
              "SP": 0,
              "SS": 0,
          }]):

    if not prev_opponent_play:
        prev_opponent_play = 'R'
    opponent_history.append(prev_opponent_play)

    last_two = "".join(opponent_history[-2:])
    if len(last_two) == 2:
        play_order[0][last_two] += 1

    potential_plays = [
        prev_opponent_play + "R",
        prev_opponent_play + "P",
        prev_opponent_play + "S",
    ]

    sub_order = {
        k: play_order[0][k]
        for k in potential_plays if k in play_order[0]
    }

    prediction = max(sub_order, key=sub_order.get)[-1:]

    ideal_response = {'P': 'S', 'R': 'P', 'S': 'R'}
    return ideal_response[prediction]


def human(prev_opponent_play):
    play = ""
    while play not in ['R', 'P', 'S']:
        play = input("[R]ock, [P]aper, [S]cissors? ")
        print(play)
    return play


def random_player(prev_opponent_play):
    return random.choice(['R', 'P', 'S'])


## Player

In [2]:
# Rock Paper Scissors solution.
#
# The player keeps several "predictors" of the opponent's next move:
#   - Markov chains of order 1..5 built on the opponent's own history
#   - models of the three adaptive bots (they react to *our* history):
#       kris    -> counters our previous move
#       mrugesh -> counters our most frequent move in the last 10
#       abbey   -> counters its order-2 prediction of our next move
# Every predictor is scored on recent games and the best one is used.
# We then play the move that beats the predicted opponent move.

COUNTER = {"R": "P", "P": "S", "S": "R"}  # move that beats the key
MOVES = ("R", "P", "S")

MARKOV_ORDERS = (1, 2, 3, 4, 5)
WINDOW = 80  # number of recent games used to score a predictor


def _reset(opponent_history, my_history, scores, last_predictions):
    opponent_history.clear()
    my_history.clear()
    scores.clear()
    last_predictions.clear()


def _markov_predict(history, order):
    """Most frequent continuation of the last `order` moves."""
    if len(history) < order or (order and history[-1] == ""):
        return None
    pattern = tuple(history[-order:])
    counts = {}
    last_seen = {}
    for i in range(order, len(history)):
        if tuple(history[i - order:i]) == pattern:
            nxt = history[i]
            counts[nxt] = counts.get(nxt, 0) + 1
            last_seen[nxt] = i
    if not counts:
        return None
    # Highest count, most recent occurrence as tie-break.
    return max(counts, key=lambda m: (counts[m], last_seen[m]))


def _kris_predict(my_history):
    if not my_history:
        return "P"  # kris answers the initial "R" with P
    return COUNTER[my_history[-1]]


def _mrugesh_predict(my_history):
    last_ten = my_history[-10:]
    if not last_ten:
        return "R"  # mrugesh falls back to "S", we beat S with R
    most_frequent = max(MOVES, key=lambda m: (last_ten.count(m), -MOVES.index(m)))
    return COUNTER[most_frequent]


def _abbey_predict(my_history):
    """Replicate abbey's order-2 model of our moves."""
    if not my_history:
        return COUNTER["R"]
    play_order = {}
    for i in range(2, len(my_history) + 1):
        pair = my_history[i - 2] + my_history[i - 1]
        play_order[pair] = play_order.get(pair, 0) + 1
    last = my_history[-1]
    candidates = {last + m: play_order.get(last + m, 0) for m in MOVES}
    predicted_our_move = max(candidates, key=candidates.get)[-1]
    return COUNTER[predicted_our_move]


def _all_predictions(opponent_history, my_history):
    predictions = {}
    for order in MARKOV_ORDERS:
        predictions[f"markov{order}"] = _markov_predict(opponent_history, order)
    predictions["kris"] = _kris_predict(my_history)
    predictions["mrugesh"] = _mrugesh_predict(my_history)
    predictions["abbey"] = _abbey_predict(my_history)
    return predictions


def player(prev_play, opponent_history=[], my_history=[], scores={},
           last_predictions={}):
    # A new match always starts with prev_play == "".
    if prev_play == "":
        _reset(opponent_history, my_history, scores, last_predictions)
    else:
        opponent_history.append(prev_play)
        # Score the prediction that was made on the previous turn.
        for name, prediction in last_predictions.items():
            if prediction is not None:
                history = scores.setdefault(name, [])
                history.append(1 if prediction == prev_play else 0)
                if len(history) > WINDOW:
                    del history[0]

    predictions = _all_predictions(opponent_history, my_history)
    last_predictions.clear()
    last_predictions.update(predictions)

    def score(name):
        history = scores.get(name, [])
        # Optimistic prior so untested predictors get a chance.
        return (sum(history) + 1) / (len(history) + 2)

    best = max(predictions, key=lambda name: (score(name), predictions[name] is not None))

    if predictions.get(best) is None:
        # No usable pattern yet: beat the opponent's most common move.
        if opponent_history:
            expected = max(MOVES, key=opponent_history.count)
        else:
            expected = "R"
    else:
        expected = predictions[best]

    move = COUNTER[expected]
    my_history.append(move)
    return move


## Test against the four bots

In [3]:
for name, bot in [('quincy', quincy), ('abbey', abbey),
                  ('kris', kris), ('mrugesh', mrugesh)]:
    rate = play(player, bot, 1000)
    print(f'{name}: {rate:.1f}% - ' + ('PASS' if rate >= 60 else 'FAIL'))


Final results: {'p1': 995, 'p2': 2, 'tie': 3}
Player 1 win rate: 99.79939819458376%
quincy: 99.8% - PASS


Final results: {'p1': 758, 'p2': 135, 'tie': 107}
Player 1 win rate: 84.88241881298993%
abbey: 84.9% - PASS


Final results: {'p1': 1000, 'p2': 0, 'tie': 0}
Player 1 win rate: 100.0%
kris: 100.0% - PASS


Final results: {'p1': 993, 'p2': 6, 'tie': 1}
Player 1 win rate: 99.3993993993994%
mrugesh: 99.4% - PASS
